# Entrega 1 — EDA y Baseline
## Sistema de Recomendación de Películas: MovieLens 100K

**Curso:** Aprendizaje de Máquina Aplicado — EAFIT  
**Profesor:** Marco Teran  
**Dataset:** MovieLens 100K — GroupLens Research  
**Fecha límite:** 23/04/2026  

---

### Pregunta central
> ¿Es posible predecir el rating (1–5) que un usuario le daría a una película que no ha visto?

In [7]:
import warnings

# Data manipulatio
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries loaded successfully ✓")

Libraries loaded successfully ✓


In [8]:
# Path to raw data
DATA_PATH = '../data/raw/ml-100k/'

# Ratings
ratings = pd.read_csv(
    DATA_PATH + 'u.data',
    sep='\t',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

# Users
users = pd.read_csv(
    DATA_PATH + 'u.user',
    sep='|',
    names=['user_id', 'age', 'gender', 'occupation', 'zip_code']
)

# Movies
movie_cols = [
    'item_id', 'title', 'release_date', 'video_release_date', 'imdb_url',
    'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

movies = pd.read_csv(
    DATA_PATH + 'u.item',
    sep='|',
    encoding='latin-1',
    names=movie_cols
)

print(f"Ratings:  {ratings.shape[0]:,} rows × {ratings.shape[1]} columns")
print(f"Users:    {users.shape[0]:,} rows × {users.shape[1]} columns")
print(f"Movies:   {movies.shape[0]:,} rows × {movies.shape[1]} columns")

Ratings:  100,000 rows × 4 columns
Users:    943 rows × 5 columns
Movies:   1,682 rows × 24 columns


In [9]:
ratings.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [10]:
users.head()

,user_id,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [11]:
movies.head()

,item_id,title,release_date,video_release_date,imdb_url,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [14]:
# --- Ratings ---
print("=" * 45)
print("RATINGS")
print("=" * 45)
print(ratings.isnull().sum())
print(f"\nDuplicates: {ratings.duplicated().sum()}")
print(f"Rating range: {ratings['rating'].min()} – {ratings['rating'].max()}")

# --- Users ---
print("\n" + "=" * 45)
print("USERS")
print("=" * 45)
print(users.isnull().sum())

# --- Movies ---
print("\n" + "=" * 45)
print("MOVIES")
print("=" * 45)
print(movies.isnull().sum())

RATINGS
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Duplicates: 0
Rating range: 1 – 5

USERS
user_id       0
age           0
gender        0
occupation    0
zip_code      0
dtype: int64

MOVIES
item_id                  0
title                    0
release_date             1
video_release_date    1682
imdb_url                 3
unknown                  0
Action                   0
Adventure                0
Animation                0
Children                 0
Comedy                   0
Crime                    0
Documentary              0
Drama                    0
Fantasy                  0
Film-Noir                0
Horror                   0
Musical                  0
Mystery                  0
Romance                  0
Sci-Fi                   0
Thriller                 0
War                      0
Western                  0
dtype: int64


## Hallazgos de calidad de datos

- **Ratings**: sin nulos, sin duplicados, rango válido (1–5). ✓
- **Users**: sin nulos en ninguna columna. ✓
- **Movies**:
  - `video_release_date`: 1,682/1,682 nulos → se descarta completamente.
  - `imdb_url`: sin valor predictivo → se descarta.
  - `release_date`: 1 nulo → se conserva, se tratará al extraer el año.
  - Columnas de géneros: sin nulos, codificación binaria correcta. ✓

In [15]:
# Drop columns with no predictive value
movies.drop(columns=['video_release_date', 'imdb_url'], inplace=True)

print(f"Movies columns after cleaning: {movies.shape[1]}")
print(movies.columns.tolist())

Movies columns after cleaning: 22
['item_id', 'title', 'release_date', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [17]:
# Extract release year from release_date
movies['release_year'] = pd.to_datetime(
    movies['release_date'], errors='coerce'
).dt.year

# Check the result
print(movies[['title', 'release_date', 'release_year']].head(10))
print(f"\nNulls in release_year: {movies['release_year'].isnull().sum()}")
print(f"Year range: {movies['release_year'].min():.0f} – {movies['release_year'].max():.0f}")

                                               title release_date  \
0                                   Toy Story (1995)  01-Jan-1995   
1                                   GoldenEye (1995)  01-Jan-1995   
2                                  Four Rooms (1995)  01-Jan-1995   
3                                  Get Shorty (1995)  01-Jan-1995   
4                                     Copycat (1995)  01-Jan-1995   
5  Shanghai Triad (Yao a yao yao dao waipo qiao) ...  01-Jan-1995   
6                              Twelve Monkeys (1995)  01-Jan-1995   
7                                        Babe (1995)  01-Jan-1995   
8                            Dead Man Walking (1995)  01-Jan-1995   
9                                 Richard III (1995)  22-Jan-1996   

   release_year  
0        1995.0  
1        1995.0  
2        1995.0  
3        1995.0  
4        1995.0  
5        1995.0  
6        1995.0  
7        1995.0  
8        1995.0  
9        1996.0  

Nulls in release_year: 1
Year range: 1922

In [22]:
# Fill the single null with the median year
median_year = movies['release_year'].median()
movies['release_year'] = movies['release_year'].fillna(median_year)

print(f"Null filled with median year: {median_year:.0f}")
print(f"Nulls remaining: {movies['release_year'].isnull().sum()}")

Null filled with median year: 1995
Nulls remaining: 0
